# Entrenar y evaluar los cuatro modelos de Sem1

Este notebook ejecuta una corrida local reproducible para **T5S, BARTO, mBART-50 y uMT5-Base**. Usa el Gold congelado de 400 noticias: 320 train, 40 validation y 40 test. La inferencia se realiza solo sobre test y las métricas se calculan después contra sus 40 claims de referencia.

No modifica el Gold, no consulta APIs y no añade recuperación, ranking ni multiclaim. Los pesos y resultados quedan en `artifacts/runs/`, ignorado por Git.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, *cwd.parents) if (path / 'scripts' / 'run_sem1_experiment.py').exists())
GOLD_DIR = ROOT / 'artifacts' / 'sem1_20261031' / 'final'
RUN_ID = 'sem1_20261031_400_claims_v1'
RUN_DIR = ROOT / 'artifacts' / 'runs' / RUN_ID
MODELS = ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']

TRAIN = GOLD_DIR / 'train.csv'
VALIDATION = GOLD_DIR / 'validation.csv'
TEST = GOLD_DIR / 'test.csv'

for path in (TRAIN, VALIDATION, TEST):
    assert path.is_file(), f'Falta el Gold: {path}'

print({'repo': str(ROOT), 'run_dir': str(RUN_DIR), 'models': MODELS})

## 1. Verificar el Gold antes de entrenar

La corrida se detiene si detecta un tamaño incorrecto, claims vacíos o fuga de `record_id`, `source_hash` o `claim_hash` entre particiones.

In [ ]:
splits = {name: pd.read_csv(path) for name, path in {'train': TRAIN, 'validation': VALIDATION, 'test': TEST}.items()}
assert {name: len(frame) for name, frame in splits.items()} == {'train': 320, 'validation': 40, 'test': 40}
assert all(frame['reference_claim'].fillna('').str.strip().ne('').all() for frame in splits.values())
for column in ('record_id', 'source_hash', 'claim_hash'):
    values = [set(frame[column]) for frame in splits.values()]
    assert not (values[0] & values[1] or values[0] & values[2] or values[1] & values[2]), column

pd.DataFrame([{'split': name, 'filas': len(frame), 'claims únicos': frame['claim_hash'].nunique()} for name, frame in splits.items()])

## 2. Verificar el dispositivo

PyTorch usa CUDA cuando está disponible, luego MPS en Apple Silicon y, si ninguno está activo, CPU. El dispositivo real queda registrado en `run_status.json` y en cada `training_manifest.json`.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
{'device': device, 'torch': torch.__version__, 'mps_built': torch.backends.mps.is_built()}

## 3. Ejecutar entrenamiento, inferencia y métricas

La celda es reanudable por defecto: si no existe `RUN_DIR`, inicia una corrida nueva; si ya existe, conserva los modelos con checkpoint y evaluación válidos. Para reiniciar desde cero, define otro `RUN_ID`; así no se mezclan resultados parciales.

In [ ]:
RESUME = True

command = [
    sys.executable, 'scripts/run_sem1_experiment.py',
    '--train', str(TRAIN),
    '--validation', str(VALIDATION),
    '--test', str(TEST),
    '--run-dir', str(RUN_DIR),
    '--models', *MODELS,
]
if RESUME:
    command.append('--resume')

print(' '.join(command))
subprocess.run(command, cwd=ROOT, check=True)

## 4. Revisar el estado y continuar con las tablas

Al terminar debe aparecer `status: completed` y una fila por modelo. Luego abre `02_resultados_modelos.ipynb`, conserva el mismo `RUN_ID` y ejecuta todas sus celdas para ver las tablas.

In [ ]:
status_path = RUN_DIR / 'run_status.json'
if status_path.is_file():
    status = json.loads(status_path.read_text(encoding='utf-8'))
    display(pd.DataFrame.from_dict(status['models'], orient='index'))
    print({'status': status['status'], 'device': status['device'], 'logs': str(RUN_DIR / 'logs')})
else:
    print('Aún no hay una corrida creada. Ejecuta la celda anterior.')